# The timestamp-specific positional penalty, per utterance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrfhyL/audio_model_initial_testing/blob/main/Colab_DeltaSweep.ipynb)

Corpus WER told us *that* late-placed utterances degrade with timestamps on. It cannot tell us
**which utterances**, or how the damage is distributed. This notebook measures a paired
per-utterance quantity instead.

For each utterance `m`:

```
delta_m = [ WER(m, 25s, ts on)  - WER(m, 5s, ts on)  ]
        - [ WER(m, 25s, ts off) - WER(m, 5s, ts off) ]
```

A **difference in differences**. The inner brackets are how much moving from 5 s to 25 s costs
that utterance, with and without timestamps. Subtracting removes any position effect that exists
independently of timestamps, along with every per-utterance confound that is constant across
offsets - speaker, sentence, duration, difficulty. What survives is the part of the positional
penalty that **only exists because timestamps are enabled**, isolated per utterance.

`delta_m > 0` means timestamps made the late placement worse for that utterance.

### Why 5 s and not 0 s as the baseline

Offset 0 is the one position whose mel is *not* a pure shift of the others:
`torch.stft(center=True)` reflect-pads 200 samples, so at offset 0 that padding mirrors the
utterance's own opening instead of zeros, perturbing mel frames 0-1. Offsets 5 s and 25 s are
both bit-identical shifts of each other, so the difference is clean.

### Corpus

The **held-out 300** - entries 700:1000 of the same seed-0 draw. The four-model scaling sweep used
entries 0:700, so these 300 are disjoint from it: no utterance is reused. Still speaker-balanced,
covering all 168 speakers (132 with 2 utterances, 36 with 1) and all 8 dialect regions.

300 clips x 2 offsets x 2 timestamp arms x 4 models = **4800 decodes**, ~10 min on a T4.

### What to expect

`delta_m` is **zero-inflated and heavy-tailed**. On a local `base` preview of these same 300
clips, 237 were exactly 0, 46 positive, 17 negative - and the five largest carried 69% of the
summed effect. Mean and median therefore say completely different things, so this notebook
reports prevalence (how many utterances are affected) and severity (how much) separately.

## 1. Environment

Same base settings as the scaling sweep: Colab GPU, fp16 on CUDA, batch 16, `openai-whisper`.

In [ ]:
!pip -q install openai-whisper jiwer soundfile

import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Provenance

Everything needed to reproduce or audit this run, recorded into `PROVENANCE` and written next to
the results. Six things are pinned:

1. **Package versions** - `openai-whisper`, torch, numpy, soundfile, jiwer, Python, CUDA.
2. **Code version** - SHA-256 of the three `whisper` source files that define the decode path
   (`audio.py`, `decoding.py`, `model.py`), plus the current commit of this repository. Package
   version strings are coarse; hashing the actual code that runs is not.
3. **Checkpoint hashes** - `whisper` embeds each checkpoint's expected SHA-256 in its download URL
   and verifies it on fetch. Both the expected and the recomputed on-disk digest are recorded, so
   the weights are provably the official ones.
4. **Precision and device** - fp16, CUDA, GPU model, driver.
5. **Decoding options** - the exact `DecodingOptions` used, verbatim.
6. **Corpus and reference digests** - the per-clip `sha256_audio` from `corpus_digests.json`, an
   aggregate digest over the 300 selected clips, and a digest over the *normalized reference
   transcripts*. The last one pins exactly which references were scored against **without storing
   any transcript text**.

### Licensing

TIMIT is LDC93S1: licensed, not redistributable. This notebook is written so that **no transcript
text and no audio ever reaches its printed output**, because saving a Colab notebook back to
GitHub commits its outputs. Only counts, digests and numeric scores are printed. The per-utterance
results file that is safe to commit carries no text column, and is asserted to be so in section 7;
the full record, which does contain hypotheses, is written to Drive only.

In [ ]:
import hashlib, json, os, platform, sys

DRIVE_ROOT  = "/content/drive/MyDrive/NAACL"
RESULTS_CSV = os.path.join(DRIVE_ROOT, "delta_results_full.csv")     # has text -> Drive only
SAFE_CSV    = os.path.join(DRIVE_ROOT, "delta_per_utterance.csv")    # numbers only -> git-safe
PROV_JSON   = os.path.join(DRIVE_ROOT, "delta_provenance.json")
LOCAL_AUDIO = "/content/corpus"
REPO        = "AgrfhyL/audio_model_initial_testing"

def sha256_file(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

import numpy, soundfile, jiwer, whisper

# --- 2. code version: hash the whisper source files that define the decode path ---
_wdir = os.path.dirname(whisper.__file__)
CODE_DIGESTS = {f: sha256_file(os.path.join(_wdir, f))[:16]
                for f in ("audio.py", "decoding.py", "model.py")}

# repo commit (best effort -- Colab has no git checkout)
try:
    import urllib.request
    with urllib.request.urlopen(
            f"https://api.github.com/repos/{REPO}/commits/main", timeout=10) as r:
        REPO_COMMIT = json.load(r)["sha"]
except Exception as e:
    REPO_COMMIT = f"unavailable ({type(e).__name__})"

PROVENANCE = {
    "experiment": "per-utterance timestamp-specific positional penalty (delta_m)",
    "metric": "delta_m = [WER(25s,on)-WER(5s,on)] - [WER(25s,off)-WER(5s,off)]",
    "packages": {
        "python": platform.python_version(), "openai-whisper": whisper.__version__,
        "torch": torch.__version__, "numpy": numpy.__version__,
        "soundfile": soundfile.__version__, "jiwer": getattr(jiwer, "__version__", "n/a"),
        "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    },
    "code_version": {"whisper_source_sha256_16": CODE_DIGESTS, "repo": REPO,
                     "repo_commit": REPO_COMMIT},
    "device": {"name": torch.cuda.get_device_name(0),
               "capability": ".".join(map(str, torch.cuda.get_device_capability(0))),
               "precision": "fp16 (encoder/decoder); mel computed fp32 on CPU"},
}
print(json.dumps(PROVENANCE, indent=1))

## 3. Corpus: the held-out 300

`corpus_digests.json` is ordered by the original seed-0 draw, so `[700:1000]` is exactly the slice
the scaling sweep did **not** touch. Each clip is copied to local disk (Drive's FUSE mount is far
too slow to read repeatedly), decoded, and checked against its frozen `sha256_audio` - a digest
over the decoded float32 samples, so it pins the array that actually reaches the mel.

Reference transcripts are read from the `.TXT` files in *your* TIMIT copy. They are hashed into a
reference digest and then never printed.

In [ ]:
import collections, glob, shutil, time
import numpy as np
import soundfile as sf

SLICE_LO, SLICE_HI = 700, 1000        # the held-out remainder

cands = [d for d in glob.glob(os.path.join(DRIVE_ROOT, "timit", "**", "TEST"), recursive=True)
         if os.path.isdir(os.path.join(d, "DR1"))]
assert cands, f"no TIMIT TEST/DR1 found under {DRIVE_ROOT}/timit"
TIMIT_TEST = sorted(cands, key=len)[0]

if not os.path.exists("corpus_digests.json"):
    !wget -q https://raw.githubusercontent.com/AgrfhyL/audio_model_initial_testing/main/corpus_digests.json
DIG = json.load(open("corpus_digests.json"))

def sha256_audio(a):
    return hashlib.sha256(np.ascontiguousarray(a, dtype=np.float32).tobytes()).hexdigest()

def load_reference(wav_path):
    with open(wav_path[:-4] + ".TXT") as f:
        return f.read().strip().split(None, 2)[2]

FILES, AUDIO, REFTEXT, bad = [], {}, {}, []
t0 = time.time()
for r in DIG["files"][SLICE_LO:SLICE_HI]:
    src = os.path.join(TIMIT_TEST, r["path"]); dst = os.path.join(LOCAL_AUDIO, r["path"])
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(src, dst)
    a, sr = sf.read(dst, dtype="float32")
    if sr != DIG["sample_rate"] or len(a) != r["samples"] or sha256_audio(a) != r["sha256_audio"]:
        bad.append(r["path"]); continue
    AUDIO[r["path"]] = a; REFTEXT[r["path"]] = load_reference(src); FILES.append(r)

assert not bad, f"{len(bad)} clips fail their frozen digest, e.g. {bad[:3]}"
assert len(FILES) == SLICE_HI - SLICE_LO

spk = collections.Counter(r["speaker"] for r in FILES)
print(f"held-out slice [{SLICE_LO}:{SLICE_HI}] verified in {time.time()-t0:.0f}s")
print(f"  {len(FILES)} clips | {len(spk)} speakers ({min(spk.values())}-{max(spk.values())} each)"
      f" | {sum(r['sec'] for r in FILES)/60:.1f} min")
print(f"  regions {dict(sorted(collections.Counter(r['region'] for r in FILES).items()))}")

# disjoint from the scaling sweep
assert not ({r["path"] for r in FILES} & {r["path"] for r in DIG["files"][:SLICE_LO]})
print(f"  disjoint from the scaling sweep's [0:{SLICE_LO}] slice: confirmed")

## 4. Controls, digests and the mel gate

Base settings are held identical to the scaling sweep. The corpus digest is an aggregate over the
300 selected clips' individual `sha256_audio` values; the reference digest is taken over
`path \t normalized_reference` lines, so it pins exactly what was scored **without retaining the
text**.

The mel gate is the control the whole experiment rests on: offsets 5 s and 25 s must be exact
integer-frame shifts of one another, or `delta_m` would be measuring a different spectrogram
rather than a different position.

In [ ]:
from whisper.audio import log_mel_spectrogram, N_SAMPLES, SAMPLE_RATE, HOP_LENGTH
from whisper.normalizers import EnglishTextNormalizer

MODELS      = ["tiny", "base", "small", "medium"]
OFFSETS_S   = [5, 25]                       # baseline and late placement
OFFSETS     = [s * SAMPLE_RATE for s in OFFSETS_S]
TS_ARMS     = [True, False]
BATCH       = 16

DECODE_OPTS = dict(task="transcribe", language="en", temperature=0.0, beam_size=None,
                   best_of=None, prompt=None, prefix=None, fp16=True,
                   suppress_blank=True, suppress_tokens="-1", max_initial_timestamp=1.0)

normalizer = EnglishTextNormalizer()
REF = {p: normalizer(t) for p, t in REFTEXT.items()}
assert all(v.strip() for v in REF.values())

# --- digests: corpus and references (no text retained) ---
corpus_digest = sha256_bytes("\n".join(
    f"{r['path']}\t{r['sha256_audio']}" for r in FILES).encode())
reference_digest = sha256_bytes("\n".join(
    f"{r['path']}\t{REF[r['path']]}" for r in FILES).encode())

PROVENANCE["decoding"] = {**DECODE_OPTS, "greedy": True,
                          "condition_on_previous_text": "N/A - decode(); prompt=None",
                          "offsets_s": OFFSETS_S, "batch": BATCH}
PROVENANCE["corpus"] = {
    "source": "TIMIT (LDC93S1) TEST -- licensed, not redistributable",
    "spec_file": "corpus_digests.json",
    "spec_sha256": sha256_file("corpus_digests.json"),
    "slice": [SLICE_LO, SLICE_HI], "n_clips": len(FILES),
    "n_speakers": len({r["speaker"] for r in FILES}),
    "corpus_digest_sha256": corpus_digest,
    "reference_digest_sha256": reference_digest,
    "reference_normalizer": "whisper EnglishTextNormalizer",
    "note": "digests pin the audio arrays and the normalized references; no text is stored",
}
print("corpus digest   ", corpus_digest)
print("reference digest", reference_digest)

# --- placement + mel gate ---
def place(audio, off_samples):
    buf = np.zeros(N_SAMPLES, dtype=np.float32)
    buf[off_samples:off_samples + len(audio)] = audio
    return buf

def mel_of(audio, off_samples, n_mels):
    return log_mel_spectrogram(torch.from_numpy(place(audio, off_samples)), n_mels)

for s, n in zip(OFFSETS_S, OFFSETS):
    assert n % HOP_LENGTH == 0
for r in FILES:
    for off in OFFSETS:
        assert off + r["samples"] <= N_SAMPLES, (r["path"], off)

import random as _random
FPS, NF = SAMPLE_RATE // HOP_LENGTH, N_SAMPLES // HOP_LENGTH
gate = 0.0
for r in _random.Random(7).sample(FILES, 40):
    a = AUDIO[r["path"]]; nf = int(np.ceil(len(a) / HOP_LENGTH)) + 2
    base = mel_of(a, 5 * SAMPLE_RATE, 80).numpy()
    seg = mel_of(a, 25 * SAMPLE_RATE, 80).numpy()
    n = min(nf, NF - 25 * FPS)
    gate = max(gate, float(np.abs(seg[:, 25*FPS:25*FPS+n] - base[:, 5*FPS:5*FPS+n]).max()))
assert gate == 0.0, f"mel gate FAILED: 5s and 25s differ by {gate:.3e}"
PROVENANCE["mel_gate"] = {"offsets_compared": [5, 25], "n_sampled": 40, "max_abs_dev": gate}
print(f"mel gate: 5 s vs 25 s bit-identical on 40 sampled clips (max |dev| {gate:.1e})")

## 5. The sweep

4800 decodes. Model-outer so each checkpoint is loaded once; its SHA-256 is recorded against the
expected digest from the download URL at load time. Both timestamp arms decode off one shared mel,
which guarantees they see bit-identical input.

Resumable: rows append to Drive and any `(model, path, offset, arm)` already present is skipped.

In [ ]:
import csv, gc

FULL_FIELDS = ["model", "n_mels", "path", "speaker", "offset_s", "timestamps",
               "text", "avg_logprob", "no_speech_prob"]

def checkpoint_digest(name):
    '''Expected SHA-256 (embedded in whisper's download URL) and the on-disk digest.'''
    url = whisper._MODELS[name]
    expected = url.split("/")[-2]
    root = os.path.join(os.getenv("XDG_CACHE_HOME",
                                  os.path.join(os.path.expanduser("~"), ".cache")), "whisper")
    p = os.path.join(root, os.path.basename(url))
    return expected, (sha256_file(p) if os.path.exists(p) else None)

done = set()
if os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, newline="") as f:
        for row in csv.DictReader(f):
            done.add((row["model"], row["path"], int(row["offset_s"]),
                      row["timestamps"] == "on"))
total = len(MODELS) * len(FILES) * len(OFFSETS_S) * len(TS_ARMS)
print(f"{len(done)}/{total} cells already done")

os.makedirs(DRIVE_ROOT, exist_ok=True)
ckpt, t_start = {}, time.time()
with open(RESULTS_CSV, "a", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=FULL_FIELDS)
    if not done:
        w.writeheader()

    for name in MODELS:
        model = whisper.load_model(name, device="cuda")
        exp, act = checkpoint_digest(name)
        ckpt[name] = {"expected_sha256": exp, "ondisk_sha256": act, "verified": exp == act,
                      "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
                      "n_mels": model.dims.n_mels,
                      "n_audio_layer": model.dims.n_audio_layer,
                      "n_text_layer": model.dims.n_text_layer}
        assert ckpt[name]["verified"], f"{name}: checkpoint digest mismatch"
        n_mels = model.dims.n_mels
        print(f"\n{name}: {ckpt[name]['params_M']}M params, n_mels={n_mels}, "
              f"sha256 {act[:16]}... verified")

        for s, off in zip(OFFSETS_S, OFFSETS):
            pending = [r for r in FILES
                       if any((name, r["path"], s, t) not in done for t in TS_ARMS)]
            if not pending:
                continue
            t0 = time.time()
            for i in range(0, len(pending), BATCH):
                batch = pending[i:i + BATCH]
                mel = torch.stack([mel_of(AUDIO[r["path"]], off, n_mels)
                                   for r in batch]).to("cuda")
                for ts_on in TS_ARMS:
                    if all((name, r["path"], s, ts_on) in done for r in batch):
                        continue
                    res = whisper.decode(model, mel, whisper.DecodingOptions(
                        **DECODE_OPTS, without_timestamps=not ts_on))
                    for r, d in zip(batch, res):
                        if (name, r["path"], s, ts_on) in done:
                            continue
                        w.writerow({"model": name, "n_mels": n_mels, "path": r["path"],
                                    "speaker": r["speaker"], "offset_s": s,
                                    "timestamps": "on" if ts_on else "off", "text": d.text,
                                    "avg_logprob": f"{d.avg_logprob:.6f}",
                                    "no_speech_prob": f"{d.no_speech_prob:.6f}"})
                        done.add((name, r["path"], s, ts_on))
                fh.flush()
            print(f"  offset {s:2d}s: {len(pending)} clips x 2 arms in {time.time()-t0:.0f}s")

        del model; gc.collect(); torch.cuda.empty_cache()

PROVENANCE["checkpoints"] = ckpt
print(f"\nsweep complete: {len(done)} cells in {(time.time()-t_start)/60:.1f} min")

## 6. `delta_m`

Per-utterance WER uses `jiwer.process_words` on a single reference/hypothesis pair, so it can
exceed 1.0 when the model emits more words than the reference - which is exactly what happens in
the runaway cases this metric is built to surface.

Reported three ways, because the distribution is zero-inflated and heavy-tailed and no single
statistic is honest on its own:

- **prevalence** - how many utterances have `delta_m != 0`, split by sign
- **severity** - mean over all clips, and mean over the affected ones only
- **concentration** - what share of the summed positive effect the worst few carry

In [ ]:
from jiwer import process_words

rows = list(csv.DictReader(open(RESULTS_CSV, newline="")))
HYP = {(r["model"], r["path"], int(r["offset_s"]), r["timestamps"]): r["text"] for r in rows}

def wer1(model, path, off, ts):
    return process_words([REF[path]], [normalizer(HYP[(model, path, off, ts)])]).wer

per_utt, summary = [], {}
for name in MODELS:
    d = []
    for r in FILES:
        p = r["path"]
        on  = wer1(name, p, 25, "on")  - wer1(name, p, 5, "on")
        off = wer1(name, p, 25, "off") - wer1(name, p, 5, "off")
        per_utt.append({"model": name, "path": p, "speaker": r["speaker"],
                        "wer_on_5":  f"{wer1(name,p,5,'on'):.6f}",
                        "wer_on_25": f"{wer1(name,p,25,'on'):.6f}",
                        "wer_off_5": f"{wer1(name,p,5,'off'):.6f}",
                        "wer_off_25":f"{wer1(name,p,25,'off'):.6f}",
                        "diff_on": f"{on:.6f}", "diff_off": f"{off:.6f}",
                        "delta_m": f"{on-off:.6f}"})
        d.append(on - off)
    d = np.array(d)
    pos, neg, zero = (d > 1e-9), (d < -1e-9), (np.abs(d) <= 1e-9)
    top5 = np.sort(d[pos])[::-1][:5].sum() if pos.any() else 0.0
    summary[name] = {
        "mean": d.mean(), "median": float(np.median(d)), "std": d.std(ddof=1),
        "n_pos": int(pos.sum()), "n_neg": int(neg.sum()), "n_zero": int(zero.sum()),
        "mean_affected": float(d[~zero].mean()) if (~zero).any() else 0.0,
        "top5_share": float(top5 / d[pos].sum()) if pos.any() else float("nan"),
        "max": float(d.max()),
    }

PARAMS = {"tiny": 39, "base": 74, "small": 244, "medium": 769}
print(f"delta_m over {len(FILES)} held-out utterances\n")
print(f"{'model':>8}{'params':>8}{'mean':>10}{'median':>9}{'pos':>6}{'zero':>6}{'neg':>6}"
      f"{'mean|affected':>15}{'max':>9}{'top5 share':>12}")
print("-" * 89)
for name in MODELS:
    s = summary[name]
    print(f"{name:>8}{PARAMS[name]:>7}M{s['mean']:>+10.4f}{s['median']:>+9.4f}"
          f"{s['n_pos']:>6}{s['n_zero']:>6}{s['n_neg']:>6}"
          f"{s['mean_affected']:>+15.4f}{s['max']:>+9.3f}{s['top5_share']:>11.0%}")

## 7. Write results, with the licensing guard

Two files. The full record carries hypothesis text and stays on Drive. The per-utterance file is
numbers and paths only - no transcript, no hypothesis - and is asserted to be so before writing,
so it is safe to commit.

In [ ]:
SAFE_FIELDS = ["model", "path", "speaker", "wer_on_5", "wer_on_25", "wer_off_5", "wer_off_25",
               "diff_on", "diff_off", "delta_m"]
FORBIDDEN = {"text", "reference", "hypothesis", "ref", "hyp", "transcript"}

assert not (set(SAFE_FIELDS) & FORBIDDEN), "a text-bearing column leaked into SAFE_FIELDS"
assert set(per_utt[0]) == set(SAFE_FIELDS), set(per_utt[0]) ^ set(SAFE_FIELDS)
for row in per_utt:                      # nothing free-text in any value
    for k, v in row.items():
        assert k in ("model", "path", "speaker") or " " not in str(v), (k, v)

with open(SAFE_CSV, "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=SAFE_FIELDS); wr.writeheader(); wr.writerows(per_utt)

PROVENANCE["outputs"] = {
    "full_results": {"path": RESULTS_CSV, "contains_text": True,
                     "sha256": sha256_file(RESULTS_CSV), "git_safe": False},
    "per_utterance": {"path": SAFE_CSV, "contains_text": False,
                      "sha256": sha256_file(SAFE_CSV), "git_safe": True},
}
PROVENANCE["summary"] = {k: {kk: float(vv) for kk, vv in v.items()} for k, v in summary.items()}
with open(PROV_JSON, "w") as f:
    json.dump(PROVENANCE, f, indent=1)

print(f"full record   -> {RESULTS_CSV}   (contains hypotheses; keep out of git)")
print(f"per-utterance -> {SAFE_CSV}   ({len(per_utt)} rows, numbers only; safe to commit)")
print(f"provenance    -> {PROV_JSON}")
print(f"\nchecked: no text-bearing column in the git-safe file")

## 8. Figures

Two views, because prevalence and severity answer different questions and one plot cannot carry
both honestly.

- **Severity** - mean `delta_m` per model with a bootstrap 95% CI. The CI matters here: with a
  handful of huge values driving the mean, an interval built by resampling utterances shows how
  fragile that mean is.
- **Prevalence** - how many of the 300 utterances are affected at all, split by sign. Positive
  means timestamps made the late placement worse for that utterance.

In [ ]:
import matplotlib.pyplot as plt

SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e5e5e2"
C_HURT, C_NONE, C_HELP = "#eb6834", "#c9c8c3", "#2a78d6"   # diverging + neutral midpoint

rng = np.random.default_rng(0)
boot = {}
for name in MODELS:
    d = np.array([float(r["delta_m"]) for r in per_utt if r["model"] == name])
    means = rng.choice(d, size=(4000, len(d)), replace=True).mean(axis=1)
    boot[name] = (d.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5))

# --- severity -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 4.3), dpi=160)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
ax.grid(True, axis="y", color=GRID, linewidth=1); ax.set_axisbelow(True)
x = np.arange(len(MODELS))
mid = [boot[m][0] for m in MODELS]
lo  = [boot[m][0] - boot[m][1] for m in MODELS]
hi  = [boot[m][2] - boot[m][0] for m in MODELS]
ax.axhline(0, color=INK2, linewidth=1, zorder=2)
ax.errorbar(x, mid, yerr=[lo, hi], fmt="o", color=C_HURT, markersize=8,
            markeredgecolor=SURFACE, markeredgewidth=2, linewidth=2, capsize=5, zorder=3)
ax.plot(x, mid, color=C_HURT, linewidth=2, zorder=2)
for xi, m in zip(x, MODELS):
    ax.annotate(f"{boot[m][0]:+.3f}", (xi, boot[m][2]), textcoords="offset points",
                xytext=(0, 9), ha="center", fontsize=9.5, color=INK)
ax.set_xticks(x); ax.set_xticklabels([f"{m}\n{PARAMS[m]}M" for m in MODELS])
ax.set_xlim(-0.5, len(MODELS) - 0.5)
ax.set_ylabel("mean  $\\Delta_m$", fontsize=11, color=INK2)
ax.set_title("Timestamp-specific positional penalty, mean per utterance\n"
             "(bootstrap 95% CI over 300 held-out utterances)",
             fontsize=12, color=INK, pad=12, loc="left")
ax.tick_params(colors=INK2, labelsize=10, length=0)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color(GRID); ax.spines[s].set_linewidth(1)
fig.tight_layout(); fig.savefig("delta_severity.png", bbox_inches="tight", facecolor=SURFACE)
fig.savefig(os.path.join(DRIVE_ROOT, "delta_severity.png"), bbox_inches="tight",
            facecolor=SURFACE)

# --- prevalence -----------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(7.2, 4.3), dpi=160)
fig2.patch.set_facecolor(SURFACE); ax2.set_facecolor(SURFACE)
ax2.grid(True, axis="y", color=GRID, linewidth=1); ax2.set_axisbelow(True)
npos = [summary[m]["n_pos"] for m in MODELS]
nzer = [summary[m]["n_zero"] for m in MODELS]
nneg = [summary[m]["n_neg"] for m in MODELS]
for vals, base, colour, lab in (
        (npos, [0]*len(MODELS), C_HURT, "$\\Delta_m>0$  timestamps hurt"),
        (nzer, npos, C_NONE, "$\\Delta_m=0$  no differential effect"),
        (nneg, [a+b for a, b in zip(npos, nzer)], C_HELP, "$\\Delta_m<0$  timestamps helped")):
    ax2.bar(x, vals, bottom=base, width=0.55, color=colour, label=lab,
            edgecolor=SURFACE, linewidth=2, zorder=3)          # 2px surface gap
for xi, m in zip(x, MODELS):
    ax2.annotate(f"{summary[m]['n_pos']}", (xi, summary[m]["n_pos"] / 2),
                 ha="center", va="center", fontsize=10, color="#ffffff", zorder=4)
ax2.set_xticks(x); ax2.set_xticklabels([f"{m}\n{PARAMS[m]}M" for m in MODELS])
ax2.set_xlim(-0.5, len(MODELS) - 0.5); ax2.set_ylim(0, len(FILES))
ax2.set_ylabel(f"utterances (of {len(FILES)})", fontsize=11, color=INK2)
ax2.set_title("How many utterances are affected at all", fontsize=12, color=INK,
              pad=12, loc="left")
ax2.tick_params(colors=INK2, labelsize=10, length=0)
for s in ("top", "right"): ax2.spines[s].set_visible(False)
for s in ("left", "bottom"): ax2.spines[s].set_color(GRID); ax2.spines[s].set_linewidth(1)
leg = ax2.legend(frameon=False, fontsize=9, loc="upper right")
for t in leg.get_texts(): t.set_color(INK2)
fig2.tight_layout(); fig2.savefig("delta_prevalence.png", bbox_inches="tight", facecolor=SURFACE)
fig2.savefig(os.path.join(DRIVE_ROOT, "delta_prevalence.png"), bbox_inches="tight",
             facecolor=SURFACE)

print("wrote delta_severity.png and delta_prevalence.png (+ copies on Drive)")
plt.show()

## 9. Verification

In [ ]:
ok = []

assert len(FILES) == 300 and len({r["path"] for r in FILES}) == 300
assert not ({r["path"] for r in FILES} & {r["path"] for r in DIG["files"][:SLICE_LO]})
spk = collections.Counter(r["speaker"] for r in FILES)
assert len(spk) == 168 and max(spk.values()) <= 2
ok.append("1. corpus: 300 held-out clips, 168 speakers (<=2 each), disjoint from the sweep set, "
          "every sha256_audio verified")

assert all(o % HOP_LENGTH == 0 for o in OFFSETS)
assert all(o + r["samples"] <= N_SAMPLES for r in FILES for o in OFFSETS)
assert PROVENANCE["mel_gate"]["max_abs_dev"] == 0.0
ok.append("2. offsets: multiples of 160, inside the window, 5 s vs 25 s mel bit-identical")

exp = len(MODELS) * len(FILES) * len(OFFSETS_S) * len(TS_ARMS)
assert len(rows) == exp, f"{len(rows)} rows, expected {exp}"
assert len({(r["model"], r["path"], r["offset_s"], r["timestamps"]) for r in rows}) == exp
ok.append(f"3. grid: {exp} rows, no duplicates, no gaps")

assert all(v["verified"] for v in PROVENANCE["checkpoints"].values())
ok.append("4. checkpoints: all four verified against the SHA-256 in whisper's download URL")

for r in per_utt[:50] + per_utt[-50:]:      # the arithmetic actually holds
    on  = float(r["wer_on_25"])  - float(r["wer_on_5"])
    off = float(r["wer_off_25"]) - float(r["wer_off_5"])
    assert abs(on  - float(r["diff_on"]))  < 1e-6
    assert abs(off - float(r["diff_off"])) < 1e-6
    assert abs((on - off) - float(r["delta_m"])) < 1e-6
ok.append("5. delta_m arithmetic: difference-in-differences recomputed from the stored WERs")

safe = list(csv.DictReader(open(SAFE_CSV, newline="")))
assert len(safe) == len(MODELS) * len(FILES)
assert not (set(safe[0]) & FORBIDDEN)
assert all(" " not in v for r in safe for k, v in r.items() if k not in ("model","path","speaker"))
ok.append("6. licensing: git-safe file has no text column and no free-text values; "
          "hypotheses confined to Drive")

for fn in ("delta_severity.png", "delta_prevalence.png"):
    assert os.path.getsize(fn) > 10000, fn
ok.append("7. figures: both PNGs written")

assert PROVENANCE["corpus"]["reference_digest_sha256"] and PROVENANCE["corpus"]["spec_sha256"]
assert PROVENANCE["code_version"]["whisper_source_sha256_16"]
ok.append("8. provenance: packages, code digests, checkpoints, device, decoding, "
          "corpus + reference digests all recorded")

for line in ok:
    print("PASS  " + line)
print(f"\nprovenance written to {PROV_JSON}")